# Statistical Properties of Financial Data — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — Introductory Econometrics for Finance, Ch. 1–2

---
**Key message:** Financial data look different from textbook data — and this has direct consequences for our models.

> Run each cell with **Shift+Enter**.


In [ ]:
!pip install yfinance seaborn statsmodels --quiet
import yfinance as yf
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
YELLOW='#FDE70E'; ORANGE='#FCB310'; RED='#C70101'; GREY='#4B4B4B'
print('✓ Libraries loaded.')

---
# Part 1 — Download & Descriptive Statistics
### 1.1 Get the Data

In [ ]:
TICKERS = ['AAPL','MSFT','JPM','^GSPC']
raw    = yf.download(TICKERS, start='2019-01-01', end='2024-12-31', auto_adjust=True, progress=False)
prices = raw['Close'].copy()
prices.columns = ['AAPL','MSFT','JPM','SP500']
prices = prices.dropna()
ret    = prices.pct_change().dropna() * 100  # in %
print(f'Downloaded: {len(ret)} trading days')
print(ret.head().round(4))

### 1.2 describe() — All Statistics in One Line

In [ ]:
print(ret.describe().round(4))
print('\nAnnualised volatility (daily std × √252):')
for col in ret.columns:
    print(f'  {col:<8}: {ret[col].std():.4f}% daily  →  {ret[col].std()*np.sqrt(252):.2f}% annual')

---
# Part 2 — Fat Tails & Skewness
### 2.1 Histogram vs. Normal

In [ ]:
sp = ret['SP500']
mu, sigma = sp.mean(), sp.std()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.hist(sp, bins=80, density=True, color='grey', alpha=0.7, label='Actual')
x_r = np.linspace(sp.min(), sp.max(), 300)
ax.plot(x_r, stats.norm.pdf(x_r, mu, sigma), color='black', lw=2.5, ls='--', label='Normal')
ax.set_xlabel('Daily return (%)'); ax.legend()
ax.set_title('S&P 500: Fat Tails vs. Normal', fontweight='bold')
ax.text(0.02, 0.96, f'Skew={sp.skew():.3f}\nKurt={sp.kurtosis():.3f}', transform=ax.transAxes,
        fontsize=10, va='top', bbox=dict(facecolor=YELLOW, alpha=0.9, edgecolor='none', boxstyle='round,pad=0.3'))
stats.probplot(sp, dist='norm', plot=axes[1])
axes[1].get_lines()[0].set(color='black', markersize=3, alpha=0.4)
axes[1].get_lines()[1].set(color=YELLOW, lw=2.5)
axes[1].set_title('Q-Q Plot (deviations = fat tails)', fontweight='bold')
plt.tight_layout(); plt.show()

### 2.2 Jarque-Bera Test

In [ ]:
print('Jarque-Bera Test  |  H₀: Returns are normally distributed')
print('-'*65)
for col in ret.columns:
    jb_s, jb_p = stats.jarque_bera(ret[col])
    dec = 'REJECT H₀' if jb_p < 0.05 else 'Fail to reject'
    print(f'{col:<8}: skew={ret[col].skew():+.3f}  kurt={ret[col].kurtosis():6.2f}  p={jb_p:.2e}  → {dec}')

---
# Part 3 — Stylized Facts
### 3.1 Volatility Clustering

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios':[2,1],'hspace':0.05})
axes[0].plot(prices['SP500'].index, prices['SP500'].values, color='black', lw=1.2)
axes[0].set_ylabel('S&P 500 Price'); axes[0].set_title('Volatility Clustering', fontweight='bold')
sp = ret['SP500']
colors_b = [RED if v<0 else 'black' for v in sp.values]
axes[1].bar(sp.index, sp.values, color=colors_b, width=1.0, alpha=0.75)
axes[1].axhline(0, color=GREY, lw=0.7); axes[1].set_ylabel('Daily return (%)')
plt.tight_layout(); plt.show()

### 3.2 ACF: Returns vs. Squared Returns

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_acf(ret['SP500'],    lags=20, ax=axes[0])
plot_acf(ret['SP500']**2, lags=20, ax=axes[1])
axes[0].set_title('ACF: Returns — low (unpredictable)', fontweight='bold')
axes[1].set_title('ACF: Squared Returns — high (GARCH!)', fontweight='bold')
plt.tight_layout(); plt.show()
print('Returns unpredictable, but risk IS predictable → GARCH models (Lecture 11)')

### 3.3 Rolling Volatility

In [ ]:
roll_vol = ret['SP500'].rolling(21).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(roll_vol.index, roll_vol.values, color='black', lw=1.3)
ax.fill_between(roll_vol.index, roll_vol.values, alpha=0.07, color='black')
ax.axhline(roll_vol.mean(), color=ORANGE, lw=1.8, ls='--', label=f'Mean = {roll_vol.mean():.1f}%')
ax.set_ylabel('Annualised vol (%)'); ax.legend()
ax.set_title('Rolling 1-Month Volatility (21-day, annualised)', fontweight='bold')
plt.tight_layout(); plt.show()

---
# Part 4 — Correlation
### 4.1 Correlation Matrix

In [ ]:
corr = ret.corr()
print('Correlation Matrix:'); print(corr.round(3))
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, ax=ax, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white', annot_kws={'size':13,'fontweight':'bold'})
ax.set_title('Correlation Matrix — Daily Returns', fontweight='bold')
plt.tight_layout(); plt.show()

### 4.2 Crisis vs. Normal

In [ ]:
periods = {'COVID (Feb–May 2020)':('2020-02-01','2020-05-31'), 'Normal (2022)':('2022-01-01','2022-12-31')}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (label, (s,e)) in zip(axes, periods.items()):
    mask = (ret.index>=s)&(ret.index<=e)
    sns.heatmap(ret.loc[mask].corr(), ax=ax, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', linewidths=0.5, linecolor='white', annot_kws={'size':13,'fontweight':'bold'})
    ax.set_title(label, fontweight='bold')
plt.tight_layout(); plt.show()
print('Correlations RISE during crises → diversification fails when you need it most!')

---
## Summary

| Concept | Formula | Python |
|---------|---------|--------|
| Annual vol | σ × √252 | `ret.std() * np.sqrt(252)` |
| Skewness | — | `ret.skew()` |
| Kurtosis | — | `ret.kurtosis()` |
| Jarque-Bera | — | `stats.jarque_bera(ret)` |
| Rolling vol | — | `ret.rolling(21).std() * √252` |
| Correlation | — | `ret.corr()` |

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW | HS 2026*